In [1]:
import cv2
import numpy as np
import os

In [2]:
pedestrian_path = r"C:\Users\mkrol\studia\semestr6\zaawansowane_algorytmy_wizyjne\lab02\pedestrian"
pedestrian_input = []
pedestrian_groundtruth = []

for i in range(300, 1100):
    input_name = f"in{i:06d}.jpg"
    gt_name = f"gt{i:06d}.png"
    
    img = cv2.imread(os.path.join(pedestrian_path, "input", input_name), cv2.IMREAD_GRAYSCALE)
    gt = cv2.imread(os.path.join(pedestrian_path, "groundtruth", gt_name), cv2.IMREAD_GRAYSCALE)
    
    if img is not None and gt is not None:
        pedestrian_input.append(img)
        pedestrian_groundtruth.append(gt)

print(f"Wczytano {len(pedestrian_input)} klatek.")

Wczytano 800 klatek.


In [8]:
YY, XX = pedestrian_input[0].shape
N = 60
iN = 0
BUF = np.zeros((YY, XX, N), np.uint8)

total_TP_mean = 0
total_FP_mean = 0
total_FN_mean = 0

total_TP_median = 0
total_FP_median = 0
total_FN_median = 0

for IG, GT in zip(pedestrian_input,pedestrian_groundtruth):
    BUF[:, :, iN] = IG
    iN = (iN + 1) % 60
    
    BG_model_mean = np.mean(BUF, axis=2).astype(np.uint8)
    BG_model_median = np.median(BUF, axis=2).astype(np.uint8)
    
    diff_mean = cv2.absdiff(IG, BG_model_mean)
    _, diff_mean = cv2.threshold(diff_mean, 30, 255, cv2.THRESH_BINARY)
    diff_mean = cv2.medianBlur(diff_mean, 3)
    
    diff_median = cv2.absdiff(IG, BG_model_median)
    _, diff_median = cv2.threshold(diff_median, 30, 255, cv2.THRESH_BINARY)
    diff_median = cv2.medianBlur(diff_median, 3)
    
    combined_view_1 = np.hstack((IG, BG_model_mean, BG_model_median))
    cv2.imshow("Klatka wyjsciowa | Model tla (dla mean) | Model tla (dla median)", combined_view_1)
    
    combined_view_2 = np.hstack((diff_mean, diff_median))
    cv2.imshow("Binaryzacja dla mean | Binaryzacja dla median", combined_view_2)
    
    total_TP_mean += np.sum((diff_mean == 255) & (GT == 255))
    total_FP_mean += np.sum((diff_mean == 255) & (GT == 0))
    total_FN_mean += np.sum((diff_mean == 0) & (GT == 255))
    
    total_TP_median += np.sum((diff_median == 255) & (GT == 255))
    total_FP_median += np.sum((diff_median == 255) & (GT == 0))
    total_FN_median += np.sum((diff_median == 0) & (GT == 255))
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cv2.destroyAllWindows()

F1_mean = (2 * total_TP_mean) / (2 * total_TP_mean + total_FP_mean + total_FN_mean)
F1_median = (2 * total_TP_median) / (2 * total_TP_median + total_FP_median + total_FN_median)

print("F1_mean: ", F1_mean)
print("F1_median: ", F1_median)

F1_mean:  0.20271715830089865
F1_median:  0.3088527151644724


In [11]:
YY, XX = pedestrian_input[0].shape
alpha = 0.2

BG_mean = pedestrian_input[0].astype(np.float64)
BG_median = pedestrian_input[0].copy()

total_TP_mean, total_FP_mean, total_FN_mean = 0, 0, 0
total_TP_median, total_FP_median, total_FN_median = 0, 0, 0

for i in range(1, len(pedestrian_input)):
    IG = pedestrian_input[i]
    GT = pedestrian_groundtruth[i]
    
    BG_mean = alpha * IG.astype(np.float64) + (1 - alpha) * BG_mean
    
    BG_median[IG > BG_median] += 1
    BG_median[IG < BG_median] -= 1
 
    diff_mean = cv2.absdiff(IG, BG_mean.astype(np.uint8))
    _, mask_mean = cv2.threshold(diff_mean, 30, 255, cv2.THRESH_BINARY)
    mask_mean = cv2.medianBlur(mask_mean, 3)
    
    diff_median = cv2.absdiff(IG, BG_median)
    _, mask_median = cv2.threshold(diff_median, 30, 255, cv2.THRESH_BINARY)
    mask_median = cv2.medianBlur(mask_median, 3)
    

    total_TP_mean += np.sum((mask_mean == 255) & (GT == 255))
    total_FP_mean += np.sum((mask_mean == 255) & (GT == 0))
    total_FN_mean += np.sum((mask_mean == 0) & (GT == 255))
    
    total_TP_median += np.sum((mask_median == 255) & (GT == 255))
    total_FP_median += np.sum((mask_median == 255) & (GT == 0))
    total_FN_median += np.sum((mask_median == 0) & (GT == 255))
    

    top_row = np.hstack((IG, BG_mean.astype(np.uint8), BG_median))
    bottom_row = np.hstack((GT, mask_mean, mask_median))
    
    combined = np.vstack((top_row, bottom_row))
    cv2.imshow("Gora: IG, BG_mean, BG_med | Dol: GT, Mask_mean, Mask_med", combined)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cv2.destroyAllWindows()

F1_mean = (2 * total_TP_mean) / (2 * total_TP_mean + total_FP_mean + total_FN_mean)
F1_median = (2 * total_TP_median) / (2 * total_TP_median + total_FP_median + total_FN_median)

print("F1_mean: ", F1_mean)
print("F1_median: ", F1_median)

F1_mean:  0.6257678747708363
F1_median:  0.9429144059489395


In [12]:
YY, XX = pedestrian_input[0].shape
alpha = 0.01  

BG_mean = pedestrian_input[0].astype(np.float64)
BG_median = pedestrian_input[0].copy()

mask_mean = np.zeros((YY, XX), dtype=np.uint8)
mask_median = np.zeros((YY, XX), dtype=np.uint8)

total_TP_mean, total_FP_mean, total_FN_mean = 0, 0, 0
total_TP_median, total_FP_median, total_FN_median = 0, 0, 0

for i in range(1, len(pedestrian_input)):
    IG = pedestrian_input[i]
    GT = pedestrian_groundtruth[i]
    
    is_background_mean = (mask_mean == 0)
    
    BG_mean[is_background_mean] = (alpha * IG[is_background_mean].astype(np.float64) + 
                                   (1 - alpha) * BG_mean[is_background_mean])
    
    is_background_median = (mask_median == 0)
    
    BG_median[(IG > BG_median) & is_background_median] += 1
    BG_median[(IG < BG_median) & is_background_median] -= 1

    diff_mean = cv2.absdiff(IG, BG_mean.astype(np.uint8))
    _, mask_mean = cv2.threshold(diff_mean, 30, 255, cv2.THRESH_BINARY)
    mask_mean = cv2.medianBlur(mask_mean, 3)

    diff_median = cv2.absdiff(IG, BG_median)
    _, mask_median = cv2.threshold(diff_median, 30, 255, cv2.THRESH_BINARY)
    mask_median = cv2.medianBlur(mask_median, 3)
    
    total_TP_mean += np.sum((mask_mean == 255) & (GT == 255))
    total_FP_mean += np.sum((mask_mean == 255) & (GT == 0))
    total_FN_mean += np.sum((mask_mean == 0) & (GT == 255))
        
    total_TP_median += np.sum((mask_median == 255) & (GT == 255))
    total_FP_median += np.sum((mask_median == 255) & (GT == 0))
    total_FN_median += np.sum((mask_median == 0) & (GT == 255))
    
    top_row = np.hstack((IG, BG_mean.astype(np.uint8), BG_median))
    bottom_row = np.hstack((GT, mask_mean, mask_median))
    combined = np.vstack((top_row, bottom_row))
    
    cv2.imshow("Konserwatywna: Gora(IG, BGm, BGmed) | Dol(GT, Mm, Mmed)", combined)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cv2.destroyAllWindows()

F1_mean = (2 * total_TP_mean) / (2 * total_TP_mean + total_FP_mean + total_FN_mean)
F1_median = (2 * total_TP_median) / (2 * total_TP_median + total_FP_median + total_FN_median)

print("F1_mean: ", F1_mean)
print("F1_median: ", F1_median)

F1_mean:  0.9417765835470403
F1_median:  0.941075532941486


In [13]:
fgbg_mog2 = cv2.createBackgroundSubtractorMOG2(history=500, varThreshold=16, detectShadows=False)
total_TP, total_FP, total_FN = 0, 0, 0


for i in range(len(pedestrian_input)):
    IG = pedestrian_input[i]
    GT = pedestrian_groundtruth[i]
    
    mask = fgbg_mog2.apply(IG, learningRate=-1)
    
    total_TP += np.sum((mask == 255) & (GT == 255))
    total_FP += np.sum((mask == 255) & (GT == 0))
    total_FN += np.sum((mask == 0) & (GT == 255))
    
    cv2.imshow("MOG2: Klatka vs Maska", np.hstack((IG, mask)))
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cv2.destroyAllWindows()

F1 = (2 * total_TP) / (2 * total_TP + total_FP + total_FN)
print("F1_mean: ", F1)

F1_mean:  0.6530242562419459


In [14]:
fgbg_knn = cv2.createBackgroundSubtractorKNN(history=500, dist2Threshold=400.0, detectShadows=False)
total_TP, total_FP, total_FN = 0, 0, 0

for i in range(len(pedestrian_input)):
    IG = pedestrian_input[i]
    GT = pedestrian_groundtruth[i]
    
    mask = fgbg_knn.apply(IG)
    

    total_TP += np.sum((mask == 255) & (GT == 255))
    total_FP += np.sum((mask == 255) & (GT == 0))
    total_FN += np.sum((mask == 0) & (GT == 255))
    
    cv2.imshow("KNN: Klatka vs Maska", np.hstack((IG, mask)))
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cv2.destroyAllWindows()

F1 = (2 * total_TP) / (2 * total_TP + total_FP + total_FN)
print("F1_mean: ", F1)

F1_mean:  0.5832220031387066
